# Telco Customer Churn Prediction

## Project Overview

This project develops and evaluates machine learning models for predicting
customer churn using the Kaggle Playground Series S6E3 dataset.

The primary evaluation metric is **ROC-AUC**.

The workflow includes:

- exploratory data analysis;
- preprocessing of numerical, ordinal, and nominal features;
- baseline model comparison;
- hyperparameter tuning;
- feature engineering and ablation experiments;
- evaluation on an untouched holdout set;
- final evaluation on the Kaggle leaderboard.

In [ ]:
import math
import re
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    OrdinalEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

%matplotlib inline

## 1. Data Loading and Initial Inspection

The training and test datasets are loaded from the Kaggle competition
directory. Before modeling, I inspect their structure, data types,
missing values, and duplicated rows.

In [ ]:
train_path = "data/train.csv"
test_path = "data/test.csv"

data_train = pd.read_csv(train_path)
data_test = pd.read_csv(test_path)

In [ ]:
print(f'Train Shape: {data_train.shape}')
print(f'Train Columns: {data_train.columns}')
print(f'Test Shape: {data_test.describe()}')
print(f'Test Columns: {data_test.columns}')

In [ ]:
display(data_train.head())
display(data_train.describe())

In [ ]:
print( 'Train duplicates --', data_train.duplicated().any())
print('Test duplicates --', data_test.duplicated().any())

print('Train NA values --', data_train.isna().sum().sum())
print('Test NA values --', data_test.isna().sum().sum())

In [ ]:
data_train.dtypes

### Column Name Standardization

For consistency and readability, column names are converted from
CamelCase to snake_case.

In [ ]:
def camel_to_snake(name):
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    name = name.replace(' ', '_')
    return name.lower()

for data in [data_train, data_test]:
    data.columns = [camel_to_snake(col) for col in data.columns]
    print(data.columns)

## 2. Exploratory Data Analysis

The exploratory data analysis focuses on identifying relationships between
customer characteristics and churn that may later inform feature
engineering and model selection.

In [ ]:
num_cols = [
    col for col in data_train.columns
    if pd.api.types.is_numeric_dtype(data_train[col])
    and col not in ['id', 'senior_citizen']
]

cat_cols = [
    col for col in data_train.columns
    if (
        pd.api.types.is_object_dtype(data_train[col])
        or col == 'senior_citizen'
    )
    and col != 'churn'
]

print('Numerical columns:', num_cols)
print('Categorical columns:', cat_cols)

In [ ]:
df = data_train.copy()
df.head()

### Target Distribution

The target variable is converted to binary format:

- `No` → 0
- `Yes` → 1

The class distribution is examined before modeling because substantial
class imbalance can affect both model evaluation and the interpretation
of classification metrics.

In [ ]:
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})

In [ ]:
counts = df['churn'].value_counts().sort_index()
labels = ['No', 'Yes']
colors = ["#99D57D", "#F27777"]

plt.figure(figsize=(7,7))
plt.pie(counts, 
        labels=labels,
        colors=colors,
        autopct=lambda pct: f'{pct:.2f}%\n({int(pct/100*len(df)):,})',
        startangle=90,
        textprops={'fontsize': 10, 'fontweight': 'bold', 'color': 'white'},
        explode=(0, 0.12))

plt.title('Distribution of Churn', 
          fontsize=15, fontweight='bold', pad=20)
plt.axis('equal')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(['tenure', 'monthly_charges', 'total_charges']):
    for churn_val, label, color in [
        (0, 'No Churn', '#99D57D'),
        (1, 'Churn', '#F27777')
    ]:
        data = df[df['churn'] == churn_val][col]

        axes[idx].hist(
            data,
            bins=30,
            alpha=0.7,
            label=label,
            color=color,
            density=True
        )

    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'{col} Distribution by Churn')
    axes[idx].legend()

plt.tight_layout()
plt.show()

### Numerical Feature Observations

- Churn is substantially more common among customers with short tenure.
- Customers who churn tend to have higher monthly charges.
- Total charges are strongly influenced by tenure, so their relationship
  with churn should be interpreted together with customer lifetime.

In [ ]:
n_cols = 4
n_rows = math.ceil(len(cat_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(20, 4.5 * n_rows)
)

axes = axes.flatten()

for idx, col in enumerate(cat_cols):

    churn_rate = (
        df.groupby(col, dropna=False)['churn']
        .apply(lambda x: (x == 1).mean())
        .sort_values(ascending=False)
    )

    colors = plt.cm.Set2(
        np.linspace(0, 1, len(churn_rate))
    )

    bars = axes[idx].bar(
        churn_rate.index.astype(str),
        churn_rate.values,
        color=colors
    )

    axes[idx].set_title(f'Churn Rate by {col}')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('Churn Rate')
    axes[idx].set_ylim(0, 1)
    axes[idx].tick_params(axis='x')

    axes[idx].bar_label(
        bars,
        labels=[f'{v:.1%}' for v in churn_rate.values],
        padding=3
    )


for idx in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

### Categorical Feature Observations

Several categorical features show strong differences in churn rate:

- Month-to-month contracts have substantially higher churn than
  one-year and two-year contracts.
- Fiber optic customers show higher churn than DSL customers.
- Customers without OnlineSecurity or TechSupport churn more frequently.
- Payment method also appears to be associated with churn.

These patterns suggest that contract type, internet service, and
support-related services may contain useful predictive information and
potential interactions.

In [ ]:
lable_enc_cols = ['gender', 'partner', 'dependents', 'phone_service', 'paperless_billing']

one_hot_enc_cols = [
  'multiple_lines', 'internet_service', 'online_security', 'online_backup',
  'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
  'payment_method'
]


df_enc = df.copy()
contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df_enc['contract'] = df_enc['contract'].map(contract_mapping)

le = LabelEncoder()

for col in lable_enc_cols:
    df_enc[col] = le.fit_transform(df[col])

df_enc = pd.get_dummies(df_enc, columns=one_hot_enc_cols, dtype = int)

In [ ]:
plt.figure(figsize=(15, 10))
corr = df_enc.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.show()

In [ ]:
churn_by_tenure = df.groupby('tenure')['churn'].mean()

plt.figure(figsize=(12, 4))
plt.plot(
    churn_by_tenure.index,
    churn_by_tenure.values,
    color='#DB0B44'
)

plt.xlabel('Tenure (months)')
plt.ylabel('Churn Rate')
plt.title('Churn Rate by Tenure')

plt.show()

Churn decreases sharply during the early stages of the customer
relationship and continues to decline with tenure. This suggests a
non-linear relationship between tenure and churn, which will later be
tested through engineered tenure features.

### EDA Summary

The exploratory analysis suggests several potentially important
predictors of churn:

- customers on month-to-month contracts churn substantially more often;
- churn decreases as customer tenure increases;
- fiber optic customers show higher churn than DSL customers;
- customers without OnlineSecurity and TechSupport churn more frequently;
- higher monthly charges are associated with higher churn.

These observations motivate later experiments with tenure transformations,
service-related features, and categorical interactions.

## 3. Data Preparation for Modeling

Before model training, the identifier column is removed and the train data is
split into training and test sets.

The test set is kept untouched during model tuning sections. It is used only once for the final
evaluation of the selected models.

In [ ]:
df.drop(columns='id', inplace=True)

y = df['churn']
X = df.drop(columns='churn')

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
y = df['churn']
X = df.drop(columns='churn')

In [ ]:
print(X.shape, y.shape)

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5

scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=RANDOM_STATE,
    test_size=0.2,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

### Preprocessing Strategy

Features are divided into three groups:

- **Numerical features** are imputed and standardized.
- **Binary/ordered categorical features** are encoded with
  `OrdinalEncoder`.
- **Nominal categorical features** are encoded using one-hot encoding.

All preprocessing steps are placed inside a scikit-learn `Pipeline`
and `ColumnTransformer` to prevent information leakage during
cross-validation. Also, a `SimpleImputer` is used purely out of consistency, even though no missing values were found.

In [ ]:
ordinal_cols = [
    'contract',
    'gender',
    'partner',
    'dependents',
    'phone_service',
    'paperless_billing',
    'senior_citizen'
]

one_hot_cols = [
    'multiple_lines',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'payment_method'
]

In [ ]:
print('Checking ordinal columns')
for i in ordinal_cols:
    print(df[i].unique())

print('Checking nominal columns')
for i in one_hot_cols:
    print(df[i].unique())

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    (
        'encoder',
        OrdinalEncoder(
            categories=[
                ['Month-to-month', 'One year', 'Two year'],
                ['Female', 'Male'],
                ['No', 'Yes'],
                ['No', 'Yes'],
                ['No', 'Yes'],
                ['No', 'Yes'],
                [0, 1]
                
            ],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )
    )
])


one_hot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('nominal', one_hot_pipeline, one_hot_cols)
])

## 4. Baseline Model Comparison

Before hyperparameter tuning and feature engineering, several baseline
models are evaluated using the same preprocessing pipeline and
cross-validation strategy.

The following models are compared:

- Logistic Regression
- Random Forest
- Gradient Boosting
- XGBoost

**ROC-AUC** is used as the primary model-selection metric. Accuracy,
precision, recall, and F1-score are reported as secondary metrics to
provide a broader view of classification performance.

All baseline models are evaluated using the same 5-fold stratified
cross-validation on the training set. The test set remains untouched.

In [ ]:
def evaluate_model(model, model_name, X=X_train, y=y_train, cv=cv):

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    output = {
        'Model': model_name,
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean()
    }
    
    return output

In [ ]:
logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE))
])

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(random_state=RANDOM_STATE, eval_metric='auc'))
])

In [ ]:
BASELINE_RESULTS = []

models = {
    'Logistic Regression': logreg_pipeline
    'Random Forest': rf_pipeline,
    'Gradient Boosting': gb_pipeline,
    'XGBoost': xgb_pipeline
}

for name, model in models.items():
    eval_result = evaluate_model(model, name)
    
    BASELINE_RESULTS.append(eval_result)


BASELINE_RESULTS = (
    pd.DataFrame(BASELINE_RESULTS)
    .sort_values(
        by='ROC-AUC',
        ascending=False
    )
    .reset_index(drop=True)
)

print(BASELINE_RESULTS)

In [ ]:
plt.figure(figsize=(8, 4))

sns.barplot(
    data=BASELINE_RESULTS,
    x='ROC-AUC',
    y='Model'
)

plt.title('Baseline Model Comparison')
plt.xlabel('Mean CV ROC-AUC')
plt.ylabel('')

plt.xlim(
    BASELINE_RESULTS['ROC-AUC'].min() - 0.01,
    BASELINE_RESULTS['ROC-AUC'].max() + 0.005
)

plt.show()

### Baseline Results

XGBoost achieved the highest baseline ROC-AUC, followed closely by
Gradient Boosting. Logistic Regression performed reasonably well despite
its simpler linear structure, while Random Forest produced the lowest
ROC-AUC among the evaluated models.

Based on these results, further hyperparameter tuning will focus on:

- Logistic Regression
- Gradient Boosting
- XGBoost

Random Forest is excluded from further experiments because it
underperformed the other models under the same cross-validation setup.

## 5. Hyperparameter Tuning

Based on the baseline comparison, further tuning focuses on
Logistic Regression, Gradient Boosting, and XGBoost.

Hyperparameters are optimized using `GridSearchCV` with the same
5-fold stratified cross-validation strategy used for baseline evaluation.
ROC-AUC remains the optimization metric.

The search is performed progressively: an initial broad search identifies
promising parameter regions, followed by narrower searches around the
best-performing configurations.

In [ ]:
def gridsearch(model, models, X=X_train, y=y_train, cv=cv):

    grid = GridSearchCV(
        estimator=models[model][0],
        param_grid=models[model][1],
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
        verbose=2
    )

    grid.fit(X, y)

    best_idx = grid.best_index_

    output = {
        'model': model,
        'best_params': grid.best_params_,
        'roc_auc_mean': grid.best_score_,
        'roc_auc_std': grid.cv_results_['std_test_score'][best_idx],
        'best_estimator': grid.best_estimator_,
        'grid': grid
    }

    return output


def make_new_grid(best_params, old_grid):
    new_grid = {}

    for param, best_value in best_params.items():

        values = sorted(old_grid[param])
        old_step = min(
            values[i+1] - values[i] 
            for i in range( len((values)) - 1)
        )

        new_step = old_step / 2

        if isinstance(best_value, int):
            new_step = max(1, int(new_step))

            new_grid[param] = [
                best_value - new_step,
                best_value,
                best_value + new_step
            ]

        else:
            new_grid[param] = [
                round(max(0.001, best_value - new_step), 4),
                best_value,
                round(best_value + new_step, 4)
            ]

    return new_grid

### First Grid Search

The first search explores the main complexity and learning-rate
parameters of each model.

Logistic Regression requires relatively little tuning, while the
tree-based boosting models benefit from a second refinement stage.
Therefore, subsequent searches focus on Gradient Boosting and XGBoost.

In [ ]:
GRID_SEARCH_RESULTS = []
GRID_1_RESULTS

models_grid = {
    'Logistic Regression': [
        logreg_pipeline,
        {
            'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
            'model__class_weight': [None, 'balanced']
        }
    ],
    'Gradient Boosting': [
        gb_pipeline,
        {
    'model__n_estimators': [100, 250, 500],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_depth': [2, 3, 4]
        }
    ],
    'XGBoost': [
        xgb_pipeline,
        {
            'model__n_estimators': [200, 400, 600],
            'model__learning_rate': [0.02, 0.05, 0.1],
            'model__max_depth': [2, 3, 4]
        }
    ]
}

In [ ]:
for name in models_grid.keys():
    
    print(f'Parameter tuning: {name} model')
    res = gridsearch(name, models_grid)
    print(res['roc_auc_mean'], res['roc_auc_std'])
    GRID_1_RESULTS.append(res)

GRID_1_RESULTS = pd.DataFrame(GRID_1_RESULTS)
print(GRID_1_RESULTS[['model', 'roc_auc_mean', 'roc_auc_std', 'best_params']])

### Second Grid Search

In [ ]:
best_params_1_grid = {
    'Gradient Boosting': GRID_1_RESULTS.loc[
        GRID_1_RESULTS['model']=='Gradient Boosting', 'best_params'
        ].iloc[0],
    'XGBoost': GRID_1_RESULTS.loc[
        GRID_1_RESULTS['model']=='XGBoost', 'best_params'
        ].iloc[0],
}

new_grid = []
for name in best_params_1_grid.keys():
    new_grid.append(
        make_new_grid(best_params_1_grid[name], models_grid[name][1])
    )

In [ ]:
GRID_2_RESULTS = []

models_grid_2 = {
    'Gradient Boosting': [
        gb_pipeline, new_grid[0]
    ],
    'XGBoost': [
        xgb_pipeline, new_grid[1]
    ]
}

In [ ]:
for name in models_grid_2.keys():
    
    print(f'Parameter tuning: {name} model')
    res = gridsearch(name, models_grid_2)
    print(res)
    GRID_2_RESULTS.append(res)

GRID_2_RESULTS = pd.DataFrame(GRID_2_RESULTS)
print(GRID_2_RESULTS[['model', 'roc_auc_mean', 'roc_auc_std', 'best_params']])

### Third Grid Search

In [ ]:
new_gb_pipeline = GRID_2_RESULTS.loc[
    GRID_2_RESULTS['model'] == 'Gradient Boosting',
    'best_estimator'
].iloc[0]

new_xgb_pipeline = GRID_2_RESULTS.loc[
    GRID_2_RESULTS['model'] == 'XGBoost',
    'best_estimator'
].iloc[0]


gb_grid_3 = {
            'model__min_samples_leaf': [5, 10, 20],
            'model__subsample': [0.7, 0.85, 1.0]
        }

xgb_grid_3 = {
            'model__min_child_weight': [1,3,5],
            'model__subsample': [0.7, 0.85, 1.0],
            'model__colsample_bytree': [0.7, 0.85, 1.0],
            'model__reg_alpha': [0, 0.01, 0.1, 1],
            'model__reg_lambda': [0.1, 1, 5, 10]
        }

In [ ]:
GRID_3_SEARCH = []

models_grid_3 = {
    'Gradient Boosting': [
        new_gb_pipeline,
        gb_grid_3
    ],
    'XGBoost': [
        new_xgb_pipeline,
        xgb_grid_3
    ]
}

In [ ]:
for name in models_grid_3.keys():
    
    print(f'Parameter tuning: {name} model')
    res = gridsearch(name, models_grid_3)
    print(res['roc_auc_mean'], res['roc_auc_std'])
    print(res['best_params'])
    GRID_3_SEARCH.append(res)

GRID_3_SEARCH = pd.concat([
    GRID_3_SEARCH.loc[GRID_3_SEARCH['model']=='Logistic Regression']
    , pd.DataFrame(GRID_3_SEARCH)
], ignore_index=True)

print(GRID_3_SEARCH[['model', 'roc_auc_mean', 'roc_auc_std', 'best_params']])

### Final Refinement

A final regularization and sampling search was performed for XGBoost
(using `min_child_weight`, `subsample`, `colsample_bytree`,
`reg_alpha`, and `reg_lambda`) and Gradient Boost (using `min_samples_leaf` and `subsample`).

The full grid for XGBoost contained 432 parameter combinations (2,160 fits with
5-fold cross-validation), making it relatively expensive to reproduce.

Best result:

- **Gradient Boosting**
- Mean CV ROC-AUC: **0.91657**
- CV standard deviation: **0.00103**
- `min_samples_leaf`: 10
- `subsample`: 0.85
 
- **Gradient Boosting**
- Mean CV ROC-AUC: **0.91629**
- CV standard deviation: **0.00092**
- `colsample_bytree`: 0.7
- `min_child_weight`: 3
- `reg_alpha`: 0.1
- `reg_lambda`: 5
- `subsample`: 0.85

### Tuned Model Selection

After progressive hyperparameter tuning, the best cross-validation
results were:

| Model | Mean ROC-AUC | CV Std |
|---|---:|---:|
| Logistic Regression | 0.90784 | 0.00106 |
| Gradient Boosting | **0.91657** | 0.00104 |
| XGBoost | 0.91629 | 0.00092 |

Hyperparameter tuning substantially improved the tree-based models,
with Gradient Boosting achieving the highest mean cross-validation
ROC-AUC.

Gradient Boosting and XGBoost remain very close in performance.
Both models are therefore retained for feature-engineering experiments,
along with Logistic Regression as a linear benchmark.

The next section investigates whether engineered features can provide
additional predictive information beyond the original feature set.

In [ ]:
logreg_final = LogisticRegression(
              max_iter=1000,
              random_state=RANDOM_STATE,
              C=10,
              class_weight=None)


gb_final = GradientBoostingClassifier(random_state=RANDOM_STATE,
                                         learning_rate=0.1,
                                         max_depth=4,
                                         n_estimators=500,
                                         min_samples_leaf=10,
                                         subsample=0.85)

xgb_final = XGBClassifier(random_state=RANDOM_STATE,
                            eval_metric='auc',
                            learning_rate = 0.1,
                            max_depth = 4,
                            n_estimators = 700,
                            colsample_bytree=0.7,
                            min_child_weight=3,
                            reg_alpha=0.1,
                            reg_lambda=5,
                            subsample=0.85)


final_models = {
    'Logistic Regression': {'model': logreg_final, 'score_mean': 0.9078362172436604, 'score_std': 0.0010562359014407395},
    'Gradient Boosting': {'model': gb_final, 'score_mean': 0.9165722202351348, 'score_std': 0.0010366242408795872},
    'XGBoost': {'model': xgb_final, 'score_mean': 0.9162898749696572, 'score_std': 0.0009202624511859507}
}

## 6. Feature Engineering

Feature engineering is evaluated as a controlled ablation study.

Each feature group is added to the tuned models and evaluated using the
same stratified cross-validation strategy. Performance is compared with
the corresponding tuned-model baseline using the change in mean ROC-AUC.

The experiments are divided into two categories:

1. **Row-wise features** — transformations that depend only on values
   within an individual observation.
2. **Context-dependent features** — transformations that learn
   statistics from the training data and therefore must be fitted
   inside the cross-validation pipeline to prevent data leakage.

### 6.1 Row-wise Feature Engineering

The first experiments create features using only information contained
within each individual customer record. Because these transformations
do not learn statistics from other observations or from the target,
they can be safely computed before cross-validation.

In [ ]:
n_gram_cols = [
    'contract',
    'internet_service',
    'payment_method',
    'online_security',
    'tech_support',
    'paperless_billing'
]


def separate_col_types(data, ordinal_cols):

    ordinal_cols = ordinal_cols.copy()

    num_cols_fe = [
        col for col in data.columns
        if (
            pd.api.types.is_numeric_dtype(data[col])
            and col not in ordinal_cols
            and col != 'churn'
        )
    ]

    cat_cols_fe = [
        col for col in data.columns
        if (
            pd.api.types.is_object_dtype(data[col])
            or isinstance(data[col].dtype, pd.CategoricalDtype)
            or col == 'senior_citizen'
        )
        and col != 'churn'
    ]

    one_hot_cols_fe = [
        col for col in cat_cols_fe
        if col not in ordinal_cols
    ]

    return {
        'num': num_cols_fe,
        'cat': cat_cols_fe,
        'ordinal': ordinal_cols,
        'one_hot': one_hot_cols_fe
    }

def arithmetics_changes(data):
    
    data = data.copy()
    data['avg_monthly_charges'] = (data['total_charges'] / data['tenure'].clip(lower=1)).astype('float32')
    data['tenure_monthly_charges'] = (data['tenure'] * data['monthly_charges']).astype('float32')
    data['expected_total'] = data['tenure']  * data['monthly_charges']
    data['charge_difference'] = data['total_charges'] - data['expected_total']

    return data

def services_count(data):
    
    data = data.copy()

    service_cols = [
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies'
    ]

    data['services_count'] = data[service_cols].eq('Yes').sum(axis=1)

    return data    

def tenure_features(data):
    
    data = data.copy()

    data['tenure_years'] = (
        data['tenure'] // 12
    ).astype('int16')

    data['tenure_segment'] = pd.cut(
        data['tenure'],
        bins=[-1, 12, 24, 48, np.inf],
        labels=False
    ).astype('int8')

    return data

def squared_features(data):
    
    data = data.copy()

    cols = [
        'tenure',
        'monthly_charges',
        'total_charges'
    ]

    for col in cols:
        data[f'{col}_sq'] = data[col] ** 2

    return data

def log_features(data):
    
    data = data.copy()

    data['total_charges_log'] = np.log1p(
        data['total_charges'].clip(lower=0)
    )

    return data

def bi_gram_cols(data, cols=n_gram_cols):
    
    data = data.copy()
    
    bg_cols = []
    
    for c1, c2 in combinations(cols, 2):
        name = f"BG_{c1}_{c2}"
        data[name] = (data[c1].astype(str) + '_' 
                      + data[c2].astype(str)).astype('category')
        bg_cols.append(name)
    
    return data, bg_cols

def tri_gram_cols(data, cols=n_gram_cols):
    
    data = data.copy()
    
    tg_cols = []
    
    for c1, c2, c3 in combinations(cols, 3):
        name = f"TG_{c1}_{c2}_{c3}"
        data[name] = (data[c1].astype(str) + '_' 
                      + data[c2].astype(str) + '_' 
                      + data[c3].astype(str)).astype('category')
        tg_cols.append(name)

    return data, tg_cols


fe_functions = {
    'Arithmetic Features': arithmetics_changes,
    'Service Count': services_count,
    'Bi-gram': bi_gram_cols,
    'Tri-gram': tri_gram_cols
    'Tenure Features': tenure_features,
    'Squared Features': squared_features,
    'Log Features': log_features
}

In [ ]:
def evaluate_features(
    model_name,
    experiments,
    models=final_models,
    X=X_train,
    y=y_train,
    cv=cv,
    ordinal_cols=ordinal_cols,
    features=fe_functions
):

    X_train_fe = X.copy()

    for function in experiments:

        func = features[function]

        if function in ['Bi-gram', 'Tri-gram']:
            X_train_fe, n_gram_cols = func(X_train_fe)

        else:
            X_train_fe = func(X_train_fe)

    fe_cols = separate_col_types(
        X_train_fe,
        ordinal_cols
    )

    preprocessor_fe = ColumnTransformer([
        (
            'num',
            num_pipeline,
            fe_cols['num']
        ),
        (
            'ordinal',
            ordinal_pipeline,
            fe_cols['ordinal']
        ),
        (
            'nominal',
            one_hot_pipeline,
            fe_cols['one_hot']
        )
    ])

    estimator = clone(
        models[model_name]['model']
    )

    model_fe = Pipeline([
        ('preprocessor', preprocessor_fe),
        ('model', estimator)
    ])

    baseline_score = models[model_name]['score_mean']

    scores = cross_val_score(
        model_fe,
        X_train_fe,
        y,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1
    )

    return {
        'model': model_name,
        'experiment': ' + '.join(experiments),
        'score_mean': scores.mean(),
        'score_std': scores.std(),
        'delta': scores.mean() - baseline_score
    }

In [ ]:
FE_SINGLE_RESULTS = []

for model_name in final_models:
    for experiment_name in fe_functions:

        print(
            f'Conducting {experiment_name} '
            f'experiment for {model_name}...'
        )

        res = evaluate_features(
            model_name,
            [experiment_name]
        )

        FE_SINGLE_RESULTS.append(res)

FE_SINGLE_RESULTS = pd.DataFrame(FE_SINGLE_RESULTS)
print(FE_SINGLE_RESULTS.sort_values(['model', 'delta'], ascending=[True, False]))

In [ ]:
FE_COMBINED_RESULTS = []

funcs = list(fe_functions.keys())

for model in final_models.keys():
    
    print(f'Conducting ALL experiments for {model}')
    
    res = evaluate_features(model, funcs)
    
    FE_COMBINED_RESULTS.append(res)

FE_COMBINED_RESULTS = pd.DataFrame(FE_COMBINED_RESULTS)
print(FE_COMBINED_RESULTS.sort_values(['model', 'delta'], ascending=[True, False]))

### Row-wise Feature Engineering Results

The engineered features have a substantially larger effect on Logistic
Regression than on the tree-based models.

In particular, logarithmic and squared transformations improve Logistic
Regression, suggesting that some relationships between the original
numerical variables and churn are non-linear.

Gradient Boosting and XGBoost show little to no improvement from the
same transformations. This is expected because tree-based models can
already capture many non-linear relationships and feature interactions
directly from the original variables.

### 6.2 Context-dependent Features

The next feature group depends on statistics learned from the training
data, including:

- quantile-bin boundaries;
- category frequencies;
- average monthly charges within customer groups.

Unlike row-wise transformations, these features cannot be computed on
the full dataset before cross-validation without introducing information
from the validation folds, which would result in data leakage.

A custom scikit-learn transformer is therefore used so that these
statistics are learned separately inside each training fold.

In [ ]:
class ContextFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        quantile_cols=None,
        frequency_cols=None,
        n_bins=10
    ):
        self.quantile_cols = quantile_cols
        self.frequency_cols = frequency_cols
        self.n_bins = n_bins


    def fit(self, X, y=None):

        X = X.copy()
        
        quantile_cols = self.quantile_cols or []
        frequency_cols = self.frequency_cols or []

        self.bin_edges_ = {}

        for col in quantile_cols:

            _, edges = pd.qcut(
                X[col],
                q=self.n_bins,
                duplicates='drop',
                retbins=True
            )

            self.bin_edges_[col] = edges

        self.frequency_maps_ = {}

        for col in frequency_cols:

            self.frequency_maps_[col] = (
                X[col]
                .value_counts(normalize=True)
                .to_dict()
            )



        self.global_monthly_mean_ = X['monthly_charges'].mean()
        
        self.internet_monthly_mean_ = (
            X.groupby('internet_service')['monthly_charges']
            .mean()
            .to_dict()
        )

        self.contract_monthly_mean_ = (
            X.groupby('contract')['monthly_charges']
            .mean()
            .to_dict()
        )

        return self

    def transform(self, X):

        X = X.copy()


        for col in self.quantile_cols:

            edges = self.bin_edges_[col]

            values = X[col].clip(
                lower=edges[0],
                upper=edges[-1]
            )

            X[f'{col}_qbin'] = pd.cut(
                values,
                bins=edges,
                labels=False,
                include_lowest=True
            )



        for col in self.frequency_cols:

            X[f'{col}_freq'] = (
                X[col]
                .map(self.frequency_maps_[col])
                .fillna(0)
            )



        internet_mean = (
            X['internet_service']
            .map(self.internet_monthly_mean_)
            .fillna(self.global_monthly_mean_)
        )

        X['monthly_vs_internet_mean'] = (
            X['monthly_charges']
            - internet_mean
        )


        contract_mean = (
            X['contract']
            .map(self.contract_monthly_mean_)
            .fillna(self.global_monthly_mean_)
        )

        X['monthly_vs_contract_mean'] = (
            X['monthly_charges']
            - contract_mean
        )

        return X

#### Leakage-safe transformation

Placing `ContextFeatureTransformer` inside the modeling pipeline is
essential for leakage-safe cross-validation.

For every cross-validation split, the transformer learns quantile
boundaries, category frequencies, and group-level averages only from
the training fold. The learned statistics are then applied to the
corresponding validation fold.

This differs from the earlier row-wise feature transformations, which
depend only on values within an individual observation and therefore
do not learn information from other rows.

In [ ]:
def make_context_model(model_name, models=final_models):

    estimator = clone(
        models[model_name]['model']
    )

    pipeline = Pipeline([
        (
            'context_features',
            ContextFeatureTransformer(
                quantile_cols=quantile_cols,
                frequency_cols=frequency_cols,
                n_bins=10
            )
        ),

        (
            'preprocessor',
            preprocessor_context
        ),

        (
            'model',
            estimator
        )
    ])

    return pipeline

def evaluate_context_features(
    model_name,
    models=final_models,
    X=X_train,
    y=y_train,
    cv=cv
):

    model = make_context_model(
        model_name,
        models=models
    )

    scores = cross_val_score(
        model,
        X,
        y,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1
    )

    baseline_score = (
        models[model_name]['score_mean']
    )

    return {
        'model': model_name,
        'experiment': 'Context Features',
        'score_mean': scores.mean(),
        'score_std': scores.std(),
        'delta': scores.mean() - baseline_score
    }

In [ ]:
def evaluate_fe(
    model_name,
    experiments,
    use_context=False,
    models=final_models,
    X=X_train,
    y=y_train,
    cv=cv
):

    X_fe = X.copy()

    for experiment in experiments:

        func = fe_functions[experiment]

        if experiment in ['Bi-gram', 'Tri-gram']:
            X_fe, _ = func(X_fe)
        else:
            X_fe = func(X_fe)

    fe_cols = separate_col_types(
        X_fe,
        ordinal_cols
    )

    num_cols_final = fe_cols['num'].copy()


    if use_context:

        context_num_cols = [
            'tenure_qbin',
            'monthly_charges_qbin',
            'total_charges_qbin',

            'contract_freq',
            'internet_service_freq',
            'payment_method_freq',
            'online_security_freq',
            'online_backup_freq',
            'device_protection_freq',
            'tech_support_freq',
            'streaming_tv_freq',
            'streaming_movies_freq',
            'multiple_lines_freq',

            'monthly_vs_internet_mean',
            'monthly_vs_contract_mean'
        ]

        num_cols_final += context_num_cols


    preprocessor_fe = ColumnTransformer([
        (
            'num',
            num_pipeline,
            num_cols_final
        ),
        (
            'ordinal',
            ordinal_pipeline,
            fe_cols['ordinal']
        ),
        (
            'nominal',
            one_hot_pipeline,
            fe_cols['one_hot']
        )
    ])

    estimator = clone(
        models[model_name]['model']
    )


    steps = []

    if use_context:

        steps.append(
            (
                'context_features',
                ContextFeatureTransformer(
                    quantile_cols=quantile_cols,
                    frequency_cols=frequency_cols,
                    n_bins=10
                )
            )
        )

    steps.extend([
        ('preprocessor', preprocessor_fe),
        ('model', estimator)
    ])

    model = Pipeline(steps)

    scores = cross_val_score(
        model,
        X_fe,
        y,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1
    )

    experiment_name = ' + '.join(experiments)

    if use_context:
        experiment_name += ' + Context'

    baseline_score = (
        models[model_name]['score_mean']
    )

    return {
        'model': model_name,
        'experiment': experiment_name,
        'score_mean': scores.mean(),
        'score_std': scores.std(),
        'delta': scores.mean() - baseline_score
    }

In [ ]:
frequency_cols = [
    'contract',
    'internet_service',
    'payment_method',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'multiple_lines'
]

quantile_cols = [
    'tenure',
    'monthly_charges',
    'total_charges'
]

context_num_cols = [
    'tenure_qbin',
    'monthly_charges_qbin',
    'total_charges_qbin',

    'contract_freq',
    'internet_service_freq',
    'payment_method_freq',
    'online_security_freq',
    'online_backup_freq',
    'device_protection_freq',
    'tech_support_freq',
    'streaming_tv_freq',
    'streaming_movies_freq',
    'multiple_lines_freq',

    'monthly_vs_internet_mean',
    'monthly_vs_contract_mean'
]

num_cols_context = num_cols + context_num_cols

context_transformer = ContextFeatureTransformer(
    quantile_cols=quantile_cols,
    frequency_cols=frequency_cols,
    n_bins=10
)

In [ ]:
preprocessor_context = ColumnTransformer([
    (
        'num',
        num_pipeline,
        num_cols_context
    ),

    (
        'ordinal',
        ordinal_pipeline,
        ordinal_cols
    ),

    (
        'nominal',
        one_hot_pipeline,
        one_hot_cols
    )
])

In [ ]:
CONTEXT_RESULTS = []

for model_name in final_models:
    
    print(f'Evaluating for {model_name}....')
    
    result = evaluate_context_features(model_name)
    
    CONTEXT_RESULTS.append(result)

CONTEXT_RESULTS = pd.DataFrame(CONTEXT_RESULTS)
print(CONTEXT_RESULTS.sort_values('delta', ascending=False))

#### Context Feature Results

Context-dependent features improve Logistic Regression, but do not
improve either boosting model.

The Logistic Regression ROC-AUC increases by approximately 0.0014,
suggesting that distributional and group-relative information provides
additional signal that the linear model cannot easily recover from the
original features.

Gradient Boosting and XGBoost show small decreases in cross-validation
performance. As with the earlier row-wise experiments, the boosting
models appear capable of extracting most of the useful structure
directly from the original feature representation.

### 6.3 Combined Feature Experiments

The strongest individual feature groups are next evaluated in
combination.

The goal is not to maximize the number of engineered features, but to
determine whether complementary transformations provide a consistent
improvement over the tuned-model baselines.

In [ ]:
final_experiments = [

    ['Log Features'],
    ['Squared'],
    ['Tenure Features', 'Log Features'],

    ['Arithmetics', 'Log Features'],
    ['Service Count', 'Log Features'],
    ['Bi-gram', 'Log Features'],

    ['Bi-gram', 'Tenure Features', 'Log Features'],
    ['Bi-gram', 'Squared', 'Log Features'],
    ['Service Count', 'Bi-gram', 'Log Features'],

    ['Arithmetics', 'Service Count', 'Bi-gram', 'Tenure Features', 'Squared', 'Log Features'],

    ['Tri-gram', 'Tenure Features', 'Log Features']
]

In [ ]:
FE_RES_FINAL = []

for model_name in final_models:
    for experiment in final_experiments:
        for use_context in [False, True]:
            
            experiment_name = ' + '.join(experiment)
            
            if use_context:
                print(f'Evaluating {experiment_name + '+ Context'} experiment for {model_name} model.....')
            else:
                print(f'Evaluating {experiment_name} experiment WITHOUT Context for {model_name} model.....')
            
            res = evaluate_fe(model_name, experiment, use_context=use_context)
            
            print(f"Delta: {res['delta']:.6f}")
            
            FE_RES_FINAL.append(res)

FE_RES_FINAL = pd.DataFrame(FE_RES_FINAL).sort_values(
    ['model', 'delta'],ascending=[True, False])
print(FE_RES_FINAL)

### Feature Engineering Summary

Feature engineering affected the models very differently.

**Logistic Regression** benefited substantially from engineered
non-linear transformations and categorical interactions. Its mean
cross-validation ROC-AUC increased from approximately **0.9078 to
0.9114**.

**Gradient Boosting** showed virtually no improvement. Its best
feature-engineered configuration improved mean ROC-AUC by only about
**0.00003**, which is negligible relative to cross-validation
variability.

**XGBoost** did not improve with any of the tested feature sets.

These results are consistent with the characteristics of the models.
Explicit non-linear transformations and interactions expand the
representational capacity of Logistic Regression, while boosting models
can already learn many non-linear relationships and interactions
directly from the original features.

For the final evaluation, the original tuned Gradient Boosting and
XGBoost models are retained. Logistic Regression is retained as a
secondary linear benchmark with its best feature-engineered
representation.

## 7. Final Holdout Evaluation

Model selection, hyperparameter tuning, and feature-engineering
experiments were performed exclusively on the training portion of the
data using stratified cross-validation.

The holdout set has remained untouched throughout this process and is
now used for the first and only time to evaluate the selected model
configurations on unseen data.

The following candidates are evaluated:

- Logistic Regression with the selected row-wise engineered features;
- tuned Gradient Boosting with the original features;
- tuned XGBoost with the original features.

ROC-AUC remains the primary evaluation metric.

In [ ]:
def evaluate_holdout(model_dict, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test):
    results = []
    experiments = model_dict['experiments']

    if model_dict['use_original']:
        tests = [[], experiments]
    else:
        tests = [experiments]
    
    for test in tests:
        if test == []:
            name = model_dict['model'] + ' Original model'
        else:
            name = model_dict['model'] + ' ' + '+'.join(test)
        
        X_train_copy = X_train.copy()
        X_test_copy = X_test.copy()

        for experiment in test:

            func = fe_functions[experiment]

            if experiment in ['Bi-gram', 'Tri-gram']:
                X_train_copy, _ = func(X_train_copy)
                X_test_copy, _ = func(X_test_copy)

            else:
                X_train_copy = func(X_train_copy)
                X_test_copy = func(X_test_copy)

        holdout_cols = separate_col_types(X_train_copy, ordinal_cols)
        num_cols_final = holdout_cols['num'].copy()

        preprocessor_holdout = ColumnTransformer([
            ('num', num_pipeline, num_cols_final),
            ('ordinal', ordinal_pipeline, holdout_cols['ordinal']),
            ('nominal', one_hot_pipeline, holdout_cols['one_hot'])
        ])
        
        estimator = clone(model_dict['original'])

        model = Pipeline([
            ('preprocessor', preprocessor_holdout),
            ('model', estimator)
        ])

        model.fit(X_train_copy, y_train)
        y_proba = model.predict_proba(X_test_copy)[:,1]

        roc_auc = roc_auc_score(y_test, y_proba)

        res = {
            'name': name,
            'roc_auc_score': roc_auc
        }

        results.append(res)
    
    return results

In [ ]:
logreg_holdout = {
    'model': 'Logistic Regression',
    'original': final_models['Logistic Regression']['model'],
    'experiments': ['Arithmetics', 'Service Count', 'Bi-gram', 'Tenure Features'],
    'use_original': False
}

gb_holdout = {
    'model': 'Gradient Boosting',
    'original': final_models['Gradient Boosting']['model'],
    'experiments': ['Tenure Features', 'Log Features'],
    'use_original': True
}

xgb_holdout = {
    'model': 'XGB',
    'original': final_models['XGB']['model'],
    'experiments': ['Service Count', 'Log Features'],
    'use_original': True
}

In [ ]:
HOLDOUT_RESULT = []
models_test = {
    'Logistic Regression': logreg_holdout,
    'Gradient Boosting': gb_holdout,
    'XGB': xgb_holdout
}

for model_name, model_dict in models_test.items():
    print(f'Fitting {model_name} and {model_dict['experiments']} for holdout test...')

    res = evaluate_holdout(model_dict)
    print(pd.DataFrame(res))
    
    HOLDOUT_RESULT.extend(res)


HOLDOUT_RESULT = pd.DataFrame(HOLDOUT_RESULT).sort_values('roc_auc_score', ascending=False)
print(HOLDOUT_RESULT)

### Holdout Results

The holdout results are consistent with the cross-validation results.

Gradient Boosting achieves the highest holdout ROC-AUC, followed very
closely by XGBoost. Logistic Regression remains less competitive despite
benefiting from feature engineering.

Importantly, the ranking of the main candidate models is consistent
between cross-validation and the previously unseen holdout set. This
provides additional evidence that the cross-validation estimates were
representative of out-of-sample performance.

The tuned Gradient Boosting model with the original feature set is
therefore selected as the final model.

## 8. Final Model and Kaggle Submission

The final Gradient Boosting pipeline is therefore trained on all
available labeled data before generating predictions for the Kaggle
test set.

In [ ]:
submission_estimator = gb_holdout['original']

submission_preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('nominal', one_hot_pipeline, one_hot_cols)
])

submission_model = Pipeline([
    ('preprocessor', submission_preprocessor),
    ('model', submission_estimator)
])

submission_model.fit(X, y)


id_df = data_test['id']
pred = submission_model.predict_proba(data_test.drop(columns='id'))[:,1]

In [ ]:
submission = pd.DataFrame({
    'id': id_df,
    'Churn': pred
})

print('Submission shape:', submission.shape)
print('Missing values:', submission.isna().sum().sum())
print(
    'Prediction range:',
    submission['Churn'].min(),
    submission['Churn'].max()
)

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

### Kaggle Leaderboard Results

The final Gradient Boosting model achieved:

- **Private leaderboard ROC-AUC: 0.91530**
- **Public leaderboard ROC-AUC: 0.91408**

For comparison, the tuned XGBoost model achieved:

- Private leaderboard ROC-AUC: 0.91506
- Public leaderboard ROC-AUC: 0.91383

The leaderboard results are consistent with the local model-selection
process: Gradient Boosting slightly outperformed XGBoost in
cross-validation, on the holdout set, and on both Kaggle leaderboard
splits.

## 9. Conclusions
This project developed and evaluated several machine-learning approaches
for customer churn prediction using ROC-AUC as the primary metric.

The main findings are:

- Customer tenure, contract type, internet service, monthly charges, and
  support-related services showed strong relationships with churn during
  exploratory analysis.

- XGBoost achieved the strongest initial baseline performance, while
  Random Forest underperformed the other candidate models.

- Hyperparameter tuning improved the boosting models, with Gradient
  Boosting ultimately achieving the highest cross-validation ROC-AUC.

- Feature engineering had a model-dependent effect. Logistic Regression
  benefited substantially from explicit non-linear transformations and
  categorical interactions, while Gradient Boosting and XGBoost gained
  little or no additional performance.

- Context-dependent features were implemented using a custom
  scikit-learn transformer inside the modeling pipeline to avoid
  cross-validation leakage.

- The untouched holdout evaluation confirmed the cross-validation model
  ranking, with Gradient Boosting achieving the highest ROC-AUC.

- The final Gradient Boosting model achieved a private Kaggle leaderboard
  ROC-AUC of **0.91530** and a public score of **0.91408**.

Overall, the experiments demonstrate that additional model or feature
complexity does not necessarily translate into better predictive
performance. A carefully tuned Gradient Boosting model using the
original feature representation provided the strongest and most
consistent results.